# Runnable 인터페이스 기초

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `RunnablePassthrough`로 입력을 그대로 전달하거나 `.assign()`으로 새 키를 추가할 수 있어요
2. `RunnableParallel`로 여러 체인을 동시에 실행하고 결과를 딕셔너리로 받을 수 있어요
3. `RunnableLambda`로 Python 함수를 체인에 삽입해 전처리·후처리를 구현할 수 있어요
4. 세 가지 Runnable을 조합해 실무형 마케팅 파이프라인을 구성할 수 있어요

## 사전 지식

- `02_Chain.ipynb`의 파이프 연산자와 `PromptTemplate` 사용법
- `03_LCEL_basic.ipynb`의 `invoke`, `batch`, `stream` 메서드
- Python 딕셔너리, 람다 표현식

## 이전 노트북 연결

`03_LCEL_basic.ipynb`에서 체인을 다양한 방식으로 실행하는 방법을 배웠어요. 이번에는 체인 내부 데이터 흐름을 세밀하게 제어하는 세 가지 핵심 Runnable을 살펴볼게요.

아래 다이어그램은 세 Runnable의 역할과 상호관계를 보여줘요.

```mermaid
flowchart TD
    IN["원본 입력<br/>{product_name, price}"]:::input

    IN --> PT["RunnablePassthrough<br/>입력 그대로 통과"]:::process
    IN --> PA["RunnablePassthrough.assign<br/>새 키 추가<br/>discount_rate, current_time"]:::process
    PA --> RP["RunnableParallel<br/>병렬 실행"]:::process
    RP --> C1["체인 1<br/>상세설명 LLM"]:::process
    RP --> C2["체인 2<br/>한줄요약 LLM"]:::process
    RP --> C3["체인 3<br/>알림톡문구 LLM"]:::process
    C1 --> RL["RunnableLambda<br/>결과 합치기"]:::process
    C2 --> RL
    C3 --> RL
    RL --> OUT["최종 출력<br/>마케팅 패키지"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

Runnable 인터페이스(Runnable Interface)는 LangChain의 핵심 추상화 개념이에요. 모든 LangChain 컴포넌트(프롬프트, 모델, 출력 파서 등)가 이 인터페이스를 구현해요.

주요 메서드:
- `invoke()`: 단일 입력을 동기 실행
- `batch()`: 여러 입력을 배치 처리
- `stream()`: 출력을 스트리밍 방식으로 반환
- `ainvoke()`, `abatch()`, `astream()`: 비동기 버전

## 1. RunnablePassthrough

`RunnablePassthrough(러너블 패스스루)`는 입력을 변경하지 않고 그대로 전달하거나, `.assign()`으로 새 키를 추가해 전달할 수 있는 Runnable이에요.

### 왜 필요한가요?

LCEL 파이프라인에서 데이터는 한 방향으로만 흘러요. 그런데 실제로는 **원본 입력을 보존하면서 LLM 호출 결과도 함께 전달**해야 하는 경우가 많아요. `RunnablePassthrough`가 이 문제를 해결해요.

### 주요 사용법

| 사용법 | 동작 |
|--------|------|
| `RunnablePassthrough()` | 입력을 변형 없이 그대로 통과시켜요 (항등 함수처럼 동작해요) |
| `RunnablePassthrough.assign(key=fn)` | 원본 입력을 유지하면서 새로운 키-값 쌍을 동적으로 추가해요 |

> **비유**: `RunnablePassthrough`는 컨베이어 벨트 위의 물건을 다음 단계로 그대로 보내는 역할이에요. `.assign()`은 물건 위에 라벨을 붙여서 보내는 역할이에요.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

### 1.1 RunnablePassthrough() - 입력 그대로 전달

가장 기본적인 사용법이에요. 입력 딕셔너리가 변형 없이 그대로 반환돼요. 파이프라인 분기점에서 원본 데이터를 유지할 때 사용해요.

In [ ]:
# ---------------------------------------------------
# RunnablePassthrough(): 입력을 변형 없이 그대로 통과시키기
# ---------------------------------------------------
# RunnablePassthrough: 항등 함수처럼 동작해요 — 입력 = 출력
# 파이프라인 분기점에서 원본 데이터를 유지할 때 사용해요


### 1.2 RunnablePassthrough.assign() - 새 키 추가

`.assign()`은 원본 입력의 모든 키를 유지하면서 새로운 키-값 쌍을 동적으로 추가해요.

**언제 사용하면 좋을까요?**
- 체인 중간에서 원본 데이터를 보존하면서 **파생 값이나 메타데이터를 추가**할 때
- 입력 데이터를 기반으로 **동적으로 새로운 정보를 계산**할 때 (예: 할인율 계산, 타임스탬프 추가)
- 여러 단계의 체인에서 **이전 단계의 모든 정보를 유지**하면서 점진적으로 데이터를 확장할 때

In [ ]:
# ---------------------------------------------------
# RunnablePassthrough.assign()으로 원본 입력에 파생 값 추가하기
# ---------------------------------------------------
# ============================================================
# TODO: assign()에 새 키를 추가해 체인을 실행해봐요
# 힌트: lambda x: 형태로 원본 입력 딕셔너리(x)에서 값을 꺼내 계산해요
# 예상 결과: product_name, price, discount_rate, current_time이 모두 포함된 응답이 나와요
# ============================================================

from datetime import datetime

## 2. RunnableParallel

`RunnablePassthrough`로 입력을 제어하는 방법을 알아봤어요. 이제 여러 체인을 동시에 실행하는 `RunnableParallel(러너블 패러렐)`을 살펴볼게요.

`RunnableParallel`은 딕셔너리 형태로 여러 Runnable을 정의하고, 동일한 입력을 각 Runnable에 동시에 전달해요. 결과는 키-값 쌍의 딕셔너리로 반환돼요.

### 주요 특징
- 여러 LLM 호출을 동시에 실행해 총 응답 시간을 줄여요
- 각 Runnable의 결과를 키로 구분해 딕셔너리로 반환해요
- 같은 입력으로 다양한 관점의 응답을 한 번에 생성할 때 유용해요

In [ ]:
from langchain_core.runnables import RunnableParallel

# ---------------------------------------------------
# RunnableParallel: 같은 입력으로 여러 체인을 동시에 실행하기
# ---------------------------------------------------
# ============================================================
# TODO: 아래 parallel_chain을 실행하고 "장점", "단점", "활용분야" 키를 확인해요
# 힌트: parallel_chain.invoke({"topic": "원하는 주제"})
# 예상 결과: {"장점": "...", "단점": "...", "활용분야": "..."} 딕셔너리가 반환돼요
# ============================================================


### 병렬 실행의 동작 방식 이해하기

`RunnableParallel`은 여러 체인을 병렬로 실행하지만, `invoke()` 메서드는 모든 체인이 완료될 때까지 기다려요. 즉, 사용자는 가장 느린 체인이 끝날 때까지 대기해요.

**데이터 흐름**:
```
입력 {"topic": "..."} ──┬── chain1 (장점) ──┐
                        ├── chain2 (단점) ──┼── 결과 {"장점": ..., "단점": ..., "활용분야": ...}
                        └── chain3 (활용) ──┘
```

다음 셀에서 순차 실행, 병렬 실행, 비동기 실행의 시간을 직접 비교해볼게요.

In [ ]:
import time

# ---------------------------------------------------
# 순차 실행 vs 병렬 실행 vs 비동기 실행 시간 비교하기
# ---------------------------------------------------


In [ ]:
from datetime import datetime

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

# ---------------------------------------------------
# 실전 예제: Passthrough + Parallel + Lambda 조합
# 하나의 상품 입력으로 여러 채널용 마케팅 카피를 동시에 생성하기
# ---------------------------------------------------


### 실전 응용: Passthrough + Parallel + Lambda 조합

세 가지 Runnable을 조합해 **하나의 입력으로 여러 형태의 결과물을 동시에 생성**하는 파이프라인을 만들어볼게요.

위 코드 셀의 파이프라인 구조를 정리하면 다음과 같아요:

```
원본 입력 → Passthrough.assign() → RunnableParallel → Lambda(합치기)
            (할인율/시간 추가)      ├─ 상세설명 (LLM)
                                   ├─ 한줄요약 (LLM)
                                   ├─ 알림톡문구 (LLM)
                                   └─ 원본입력 (Passthrough)
```

이 패턴은 실무에서 자주 활용돼요:
- 같은 데이터로 다양한 채널용 콘텐츠를 동시에 생성할 때 (웹, 앱 푸시, 문자)
- A/B 테스트용 여러 버전의 카피를 한 번에 생성할 때
- 분석 결과를 여러 관점에서 동시에 요약할 때

## 3. RunnableLambda

`RunnableParallel`로 병렬 실행을 다뤄봤어요. 이번에는 일반 Python 함수를 체인에 삽입하는 `RunnableLambda(러너블 람다)`를 살펴볼게요.

`RunnableLambda`는 일반 Python 함수를 Runnable로 변환해 파이프라인에 포함시켜요. 체인 중간에 커스텀 로직을 삽입할 수 있어요.

### 주요 활용 사례
- 입력 전처리: 사용자 친화적 형식을 LLM 입력 형식으로 변환해요
- 출력 후처리: LLM 응답에서 특정 부분만 추출하거나 포맷팅해요
- 데이터 타입 변환: 문자열 → 딕셔너리, 딕셔너리 → 문자열 등의 변환이 필요할 때 사용해요

In [ ]:
from langchain_core.runnables import RunnableLambda

# ---------------------------------------------------
# RunnableLambda: 일반 Python 함수를 파이프라인에 삽입하기
# ---------------------------------------------------


In [ ]:
# ---------------------------------------------------
# 여러 RunnableLambda를 순서대로 연결하기 (데이터 타입 변환 포함)
# ---------------------------------------------------


In [ ]:
# ---------------------------------------------------
# RunnableLambda를 입력 전처리·출력 후처리 양쪽에 사용하기
# ---------------------------------------------------
# ============================================================
# TODO: language를 "영어"로 바꿔서 실행하고 응답이 달라지는지 확인해요
# 힌트: invoke()의 딕셔너리에서 "language" 값을 변경해요
# 예상 결과: prepare_input()이 "{language}로 {topic}에 대해 설명해줘" 문자열을 생성해요
# ============================================================


## 4. 종합 예제: 세 가지 Runnable 조합하기

`RunnablePassthrough`, `RunnableParallel`, `RunnableLambda`를 모두 배웠어요. 이번에는 세 가지를 함께 조합해 실무형 마케팅 패키지 생성 파이프라인을 만들어볼게요.

이 예제는 앞서 섹션 2에서 다룬 실전 응용과 동일한 구조이지만, 최종 출력을 딕셔너리 대신 사람이 바로 읽을 수 있는 보고서 형태의 문자열로 반환해요.

In [ ]:
from datetime import datetime

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

# ---------------------------------------------------
# 종합 예제: 세 가지 Runnable 조합 (보고서 형태 출력)
# ---------------------------------------------------


---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

| Runnable | 역할 | 주요 사용법 |
|----------|------|-------------|
| `RunnablePassthrough` | 입력을 그대로 통과시켜요 | 분기점에서 원본 데이터 유지 |
| `RunnablePassthrough.assign()` | 원본 유지 + 새 키 추가 | 파생 값(할인율, 타임스탬프) 동적 주입 |
| `RunnableParallel` | 여러 체인 동시 실행 | 같은 입력으로 다양한 출력 생성 |
| `RunnableLambda` | Python 함수를 Runnable로 변환 | 전처리·후처리·데이터 타입 변환 |

세 가지를 조합하면 `enrich → parallel → merge` 패턴으로 복잡한 실무 파이프라인을 선언적으로 표현할 수 있어요.

## 다음 노트북 예고

다음 노트북에서는 **메모리(Memory)**와 **대화 이력 관리**를 다뤄요. 지금까지 수동으로 관리하던 대화 히스토리를 LangChain 컴포넌트가 자동으로 처리하는 방법을 알아볼게요.